# A/B testing with paths: the `quick_add` experiment

> **Development version.** This notebook uses the A/B testing tools that ship with
> retentioneering 5.4.0 (`get_ab_test`, `get_ab_bootstrap`, `get_sample_ratio`, funnel
> intervals, `datasets.load_ab_test`). Until that release is on PyPI it runs against the
> development install:
>
> ```bash
> git clone https://github.com/retentioneering/retentioneering-tools
> cd retentioneering-tools
> make install && make build   # needs uv and Node.js: dependencies, then the widget bundle
> pip install -e .             # into the environment of your notebook kernel
> ```
>
> From 5.4.0 on, `pip install "retentioneering>=5.4.0"` is all it takes.

Paths generate hypotheses; experiments confirm them. A transition graph shows a loop that
exists in one arm only, a clustering finds a group that converts differently, a funnel shows a
step that leaks: each of these is a guess about cause. A randomised experiment turns a guess into
a finding, and only for the comparisons that the randomisation protects. This notebook works
through one experiment from the assignment table to the final summary and keeps the two kinds of
evidence apart: every step says what it proves and what remains a hypothesis.

**The experiment.** Northwind Home, a synthetic home-goods store, tested `quick_add`: a one-tap
"add to basket" button on catalog and search tiles. Users were randomised 50/50 at their first
app open over six weeks; the log covers eight weeks. In control an add is `add_to_cart`; in
treatment most adds are `quick_add`. The question the product team asked: does `quick_add` sell
more?

**How to read the numbers.** Every number quoted below is printed by a library call in the cell
above it; the prose only interprets it. `delta` is always treatment minus control, and intervals
are 95 %. Widgets are shown with `render_static()` so that they survive an HTML export; drop that
call to get the live widget, whose sidebar recomputes. The dataset has planted structure (listed in
`retentioneering.datasets.ab_test`); the last step compares what the analysis recovered with it.

In [ ]:
import os

os.environ["RETENTIONEERING_NO_TRACK"] = "1"

import pandas as pd
from sklearn.metrics import adjusted_rand_score

import retentioneering as rete
from retentioneering import datasets

if not hasattr(rete.Eventstream, "get_ab_test"):
    raise ImportError(
        "This notebook needs retentioneering >= 5.4.0 or the development install "
        "(see the first cell)."
    )
print("Retentioneering", rete.__version__)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:.4f}".format)

DIFF = ("exp_group", "treatment", "control")  # delta = treatment - control
SCORE_COLS = [
    "metric",
    "group1_value",
    "group2_value",
    "delta",
    "delta_ci_low",
    "delta_ci_high",
    "p_value_adjusted",
    "mde",
    "verdict",
]


def readout(row: pd.Series) -> str:
    # One line per scorecard row; 0/1 metrics in percentage points.
    scale, unit = (100, " pp") if row["test"] == "z_test" else (1, "")
    return (
        f"{row['metric']:>18}: treatment {row['group1_value'] * scale:.2f}, "
        f"control {row['group2_value'] * scale:.2f}, "
        f"delta {row['delta'] * scale:+.2f}{unit} "
        f"[{row['delta_ci_low'] * scale:+.2f}, {row['delta_ci_high'] * scale:+.2f}], "
        f"adjusted p {row['p_value_adjusted']:.3g}, MDE {row['mde'] * scale:.2f}{unit}"
        f" -> {row['verdict']}"
    )

## 1. Who is in the experiment

The population of an experiment is the **assignment table**, not the event log. A user who was
assigned and never produced a tracked event is still part of the experiment (intention to treat,
ITT); dropping such users compares only the engaged ones, and the treatment may change who that
is. `load_ab_test_assignment()` returns one row per assigned user with the arm and the assignment
time.

Production logs often do not carry the arm. The bundled log does, but we rebuild the column from
the table the way you would with your own data: `add_segment(path_values=...)` broadcasts one value
per user to all of the user's events, so the arm is constant per path by construction.

In [ ]:
stream = datasets.load_ab_test()
assignment = datasets.load_ab_test_assignment()

stream = stream.drop_segment("exp_group").add_segment(
    "exp_group", path_values=assignment["arm"]
)
overview = stream.describe(top_events=None)
print("assignment table:", len(assignment), "users")
print("event log:", overview["shape"])
print("dates:", overview["date_range"]["min"], "to", overview["date_range"]["max"])
before_assignment = int(
    (stream.df["timestamp"] < stream.df["user_id"].map(assignment["assigned_at"])).sum()
)
print("events before assignment:", before_assignment)
assignment.head()

The log holds fewer users than the table: the difference are the silent users, assigned but never
seen again. No event precedes its user's assignment, so everything in the log happened after the
user entered the experiment.

**Sample ratio mismatch (SRM)** is the first check of any experiment: if the arms are not the size
the design says, the assignment, the logging or a filter lost users from one arm, and no metric can
be trusted until that is explained. `get_sample_ratio` runs a chi-square test on the assignment
table, silent users included. `by="platform"` repeats it within the platform users signed up on,
a variable fixed at assignment (on a variable the treatment can change, such as "reached the
cart", unequal shares are an effect, not a bug).

In [ ]:
srm = stream.get_sample_ratio("exp_group", assignment=assignment, by="platform")
print(srm["basis"])
print(
    f"chi2 = {srm['chi2']:.3f}, p = {srm['p_value']:.3f}, mismatch = {srm['mismatch']}, "
    f"silent paths = {srm['silent_paths']}, paths with several arms = "
    f"{srm['multi_valued_paths']}"
)
display(srm["levels"])
srm["by_levels"]

**What this proves.** The split matches the 50/50 design overall and within each platform (the
`mismatch` flags above), no user carries two arms, and the silent users are counted, split across
both arms. **What it does not prove.** Anything about the treatment: SRM is a check of the design,
and it is the last check we can run before looking at outcomes.

## 2. The scorecard

`get_ab_test` compares the arms on several path metrics at once: per-arm estimates, the delta with
its interval, a p-value adjusted for multiplicity, the minimum detectable effect (MDE) at the
realised sample size and a verdict. With `assignment=` the population is the table: silent users
count as zeros for 0/1 metrics and totals.

**The analysis plan, fixed before looking at any outcome.** Two reads of the same metrics:

- the **primary read**: the whole observation window, every assigned user (ITT);
- the **secondary read**: the same comparison within each user's first 28 days after assignment
  (`horizon="28D"`). Users entered over six weeks and the log stops at a fixed date, so the whole
  window observes an early user for eight weeks and a late one for two; the horizon gives every
  user the same exposure age, and users whose 28 days had not closed are left out
  (`immature_paths`).

Both reads are declared here, with the horizon, before either table is printed. Trying horizons
after seeing the primary read until one turns green would be peeking. The two reads are not
corrected against each other, so the primary one carries the decision.

The metrics, chosen before looking at the data:

- `conversion`: share of users with an `order_success`, the primary metric;
- `adders`: share of users who added anything. The feature replaces `add_to_cart` with
  `quick_add` in treatment, so a metric on `add_to_cart` alone would show a large, meaningless
  drop: `has_any_event` over both events measures the behaviour, not the button;
- `orders_per_user` and `revenue_per_user`: totals per assigned user (Welch tests);
- `aov`: average order value, revenue over orders. A ratio of sums is not a per-user mean, so the
  library resamples users (path bootstrap) to get its interval.

In [ ]:
metrics = [
    {
        "name": "conversion",
        "metric": "has_event",
        "metric_args": {"event": "order_success"},
    },
    {
        "name": "adders",
        "metric": "has_any_event",
        "metric_args": {"events": ["add_to_cart", "quick_add"]},
    },
    {
        "name": "orders_per_user",
        "metric": "event_count",
        "metric_args": {"event": "order_success"},
    },
    {
        "name": "revenue_per_user",
        "metric": "column_sum",
        "metric_args": {"column": "order_value"},
    },
    {
        "name": "aov",
        "metric": "column_sum",
        "metric_args": {"column": "order_value"},
        "denominator": {
            "metric": "event_count",
            "metric_args": {"event": "order_success"},
        },
    },
]
scorecard = stream.get_ab_test(DIFF, metrics, assignment=assignment)
for _, row in scorecard.iterrows():
    print(readout(row))
scorecard[SCORE_COLS + ["group1_paths", "group2_paths", "test", "ci_basis"]]

In [ ]:
# The secondary read, declared above: the same metrics within 28 days of assignment.
scorecard_28d = stream.get_ab_test(DIFF, metrics, assignment=assignment, horizon="28D")
print("users left out (window not closed):", scorecard_28d["immature_paths"].iloc[0])
for _, row in scorecard_28d.iterrows():
    print(readout(row))
scorecard_28d[SCORE_COLS + ["group1_paths", "group2_paths", "immature_paths"]]

**The primary read.** Over the whole log every metric is **inconclusive**. Inconclusive is not
"no effect": read the interval. The conversion interval runs from about -1.1 to +2.3 pp, so an effect outside that range
is unlikely, and a smaller one, a small lift or a small loss, is consistent with what we see. The
MDE answers a different question: it is the effect this test detects with 80 % power at the
corrected alpha. It describes the design, not the true effect, so an inconclusive row does not say
that the effect is smaller than the MDE; the interval says how large it can be. The AOV row is a
bootstrap row (`ci_basis="path_bootstrap"`).

**The secondary read.** Within 28 days of assignment the test says `increase` for conversion,
orders and revenue per user (the verdicts of the second table); adders and AOV stay inconclusive.

**What this proves.** Both reads are randomised ITT comparisons of estimands declared in advance,
and each is right about its own question; neither is "the effect". Over the whole window none of
the five metrics shows a detectable difference, and the intervals say which effects remain
compatible with the data; within the first 28 days conversion, orders and revenue rose. **What
remains open.** Why the two reads differ (step 4 looks at time since assignment) and whether the
average hides users who react in opposite directions (step 5).

## 3. The funnel: overall steps versus conditional steps

`funnel(ci_level=0.95, correction="holm")` adds Wilson intervals to every bar and, in diff mode, a
Newcombe interval and a pooled z-test p-value to every **overall** step (share of the arm's users
with at least one event who reached the step; the silent users of step 1 are not in the funnel), with Holm's correction across the steps. Conditional (step-to-step) rates
get an interval but no p-value. The steps follow the purchase path; the add itself is left out
because its event differs between the arms.

In [ ]:
steps = ["product_view", "cart", "checkout_start", "payment", "order_success"]
stream.funnel(steps, diff=DIFF, ci_level=0.95, correction="holm").render_static()

In [ ]:
funnel = stream.funnel_data(steps, diff=DIFF, ci_level=0.95, correction="holm")
print(funnel["correction_family"])
print(
    "users per arm in the funnel:",
    funnel["funnel1_total_paths"],
    "treatment,",
    funnel["funnel2_total_paths"],
    "control",
)
funnel_table = pd.DataFrame(funnel["steps"]).set_index("step")
funnel_table[
    [
        "delta_conversion_rate",
        "delta_conversion_rate_ci_low",
        "delta_conversion_rate_ci_high",
        "delta_conversion_rate_p_value_adjusted",
        "step_conditional",
        "delta_step_conversion_rate",
        "delta_step_conversion_rate_ci_low",
        "delta_step_conversion_rate_ci_high",
    ]
]

Every overall step is grey after Holm: no step of the purchase path shows a detectable difference
in the share of users who reached it, and the intervals say how large one could still be. The
funnel counts users with at least one event, so its totals are smaller than the assignment table's;
for an ITT readout of a step use `get_ab_test` with `has_event`.

The conditional (step-to-step) deltas tell a different story, in opposite directions: among users
who reached the cart, more treatment users started a checkout (`cart -> checkout_start`), and among
users who started a checkout, fewer treatment users paid (`checkout_start -> payment`); see their
intervals above. The funnel deliberately gives these rows no p-value and no colour. The cell below
computes the p-values a naive tool would print, so that you recognise them when you see them
elsewhere.

In [ ]:
# What a naive tool would report: NOT funnel output.
# A pooled two-proportion z-test on the step-to-step rate, with the previous step's
# users as the denominator. The denominator is a post-exposure population.
from scipy.stats import norm

naive = []
for previous, current in zip(funnel["steps"], funnel["steps"][1:]):
    n1, n2 = previous["funnel1_unique_paths"], previous["funnel2_unique_paths"]
    x1, x2 = current["funnel1_unique_paths"], current["funnel2_unique_paths"]
    pooled = (x1 + x2) / (n1 + n2)
    se = (pooled * (1 - pooled) * (1 / n1 + 1 / n2)) ** 0.5
    z = (x1 / n1 - x2 / n2) / se
    naive.append(
        {
            "step": f"{previous['step']} -> {current['step']}",
            "naive_delta": x1 / n1 - x2 / n2,
            "naive_p_value": 2 * norm.sf(abs(z)),
        }
    )
naive = pd.DataFrame(naive)
naive["naive_p_value"] = naive["naive_p_value"].map("{:.2g}".format)
naive

The naive p-value of `checkout_start -> payment` is tiny, and it answers the wrong question. Its
denominator, "users who started a checkout", is chosen by the treatment, so treatment and control
checkout starters are different people and their payment rates can differ even if nothing changed
for anyone who meant to pay. The overall payment step, which counts every user of the arm, is grey.
A plausible mechanism from the product spec: the treatment build also shows a "checkout now"
shortcut once an add reaches the cart, after a `quick_add` or a plain `add_to_cart` alike, and it
starts checkouts that would never have been started from the cart page: more starts, and fewer of
them paid.

**What this proves.** No overall step of the purchase path shows a detectable difference.
**What remains a hypothesis.** The "checkout now" mechanism behind the two conditional deltas: the funnel
describes them, it does not test them. A test needs a metric defined on the whole arm, for example
the share of users with a checkout that did not end in payment, put into `get_ab_test`.

## 4. Over time: calendar weeks, exposure age and novelty

`period="W"` gives one ITT row per calendar week: every user assigned by the end of the week is in
the week, active or not, and the metric is "converted during this week".

In [ ]:
conversion = [metrics[0]]
weekly = stream.get_ab_test(DIFF, conversion, assignment=assignment, period="W")
weekly[
    [
        "period",
        "group1_paths",
        "group2_paths",
        "delta",
        "delta_ci_low",
        "delta_ci_high",
        "p_value_adjusted",
        "verdict",
    ]
]

The weekly delta drifts from positive in the first weeks to negative in the last ones. Calendar
weeks cannot tell why: in week one every user is days old, in the last week the early users are
two months old, so calendar time mixes the age of the user with the season. Holm's correction runs
over the weeks (`correction_family`): the weekly rows of one metric are one family.

**Why the two declared reads of step 2 differ.** The whole-window read averages each user's
whole time in the experiment, eight weeks for an early user and two for a late one; the 28-day read
averages every user's first four weeks. The gap between them suggests that the effect changes with
time since assignment.

The since-assignment rows show it. `period="14D", period_basis="since_assignment"` bins each user's
events by days since their own assignment; `horizon="42D"` keeps only users observed for all three
bins, so each bin compares the same users. This 42-day read is a sensitivity check that explains
the two declared reads, not a third estimand to report.

In [ ]:
since = stream.get_ab_test(
    DIFF,
    conversion,
    assignment=assignment,
    period="14D",
    period_basis="since_assignment",
    horizon="42D",
)
since[
    [
        "period",
        "group1_paths",
        "group2_paths",
        "immature_paths",
        "group1_value",
        "group2_value",
        "delta",
        "delta_ci_low",
        "delta_ci_high",
        "p_value_adjusted",
        "verdict",
    ]
]

The effect is positive in the first two weeks after assignment and negative four weeks later. Two
separate rows do not test that the effect *changed*: that needs the difference of the deltas with
its own interval. `get_ab_bootstrap` resamples users once and evaluates every period on the same
replicates, so a `contrast` between periods has a joint interval.

In [ ]:
replicates = stream.get_ab_bootstrap(
    DIFF,
    conversion,
    assignment=assignment,
    period="14D",
    period_basis="since_assignment",
    horizon="42D",
    contrast={"novelty": (["0-13"], ["28-41"])},
)
novelty = replicates.loc[replicates["contrast"] == "novelty", "delta"]
by_period = since.set_index("period")["delta"]
point = by_period["0-13"] - by_period["28-41"]
low, high = novelty.quantile([0.025, 0.975])
print(f"replicates: {len(novelty)}")
print(
    "novelty contrast (effect in days 0-13 minus effect in days 28-41): "
    f"{point * 100:+.2f} pp, 95% interval [{low * 100:+.2f}, {high * 100:+.2f}] pp"
)

The interval of the contrast excludes zero: the treatment effect in days 28-41 after assignment is
smaller than in days 0-13. The usual companion check, an A/A strip on the weeks before assignment
(the arms must not differ before the treatment could act), is missing here: Northwind assigns users
at their first app open, so no event precedes `assigned_at` (step 1 counted them) and there is
nothing before assignment to compare. With a product that has history, keep each user's events
before `assigned_at` in a stream of their own (a pandas filter like the count in step 1;
`truncate_paths` cannot cut at a per-user timestamp) and compare the arms on it.
`get_ab_test(assignment=...)` itself drops those events from the readout.

**What this proves.** The treatment effect on conversion depends on time since assignment: larger
early, smaller (here negative) later, with a joint interval that excludes zero, on the same users
in every bin. **What remains open.** *Why*: novelty (users try the button and lose interest) and
cannibalisation (early orders that would have happened later anyway) both predict this shape. The
long-run effect is the one that matters for a launch decision, and the late bin speaks to it more
directly than the 28-day read; but the late bin was not declared in step 2, so it explains the
declared reads rather than replacing them. If the long run is the decision, declare a late window
in the next test.

## 5. Who reacts: blind clusters on the pre-exposure window

Averages hide subgroups that move in opposite directions. To look for them without fooling
ourselves, the subgroups must be fixed **before exposure**: a group defined by post-exposure
behaviour (say, "users who used `quick_add`") is selected by the treatment and compares different
people across arms. We use the part of each path before the user's **first add**, cut out with
`truncate_paths`.

In Northwind that window is pre-exposure **by construction**: the generator fixes each user's type
before assignment and draws the browsing up to the first add identically in both arms. With real
data it is an assumption to declare, not a fact: `quick_add` sits on catalog and search tiles, so a
treated user sees it before the first add, and the treatment could change when that add happens.
The first check is the one the data can make: the window must not differ between the arms.

In [ ]:
pre_exposure_end = [
    {"pattern": "add_to_cart", "offset": -1},
    {"pattern": "quick_add", "offset": -1},
    "path_end",
]
pre_exposure = stream.truncate_paths("path_start", pre_exposure_end)

# Balance of the window between the arms: necessary for a pre-exposure window, not sufficient.
window_metrics = [
    {"name": "window_length", "metric": "length"},
    {"name": "window_active_days", "metric": "active_days"},
    {
        "name": "window_search_filter",
        "metric": "has_event",
        "metric_args": {"event": "search_filter"},
    },
    {
        "name": "window_collections_view",
        "metric": "has_event",
        "metric_args": {"event": "collections_view"},
    },
]
window_balance = pre_exposure.get_ab_test(DIFF, window_metrics, assignment=assignment)
window_balance[SCORE_COLS[:-2] + ["verdict"]]

The window has the same length, the same number of active days and the same marker events in
both arms: every row is inconclusive, with narrow intervals around zero. That is consistent with a
pre-exposure window, and it is all the data can say: a cut can be balanced and still be moved by
the treatment for some users (step 9 shows a balanced split that is post-exposure). We cluster users
on that window, pooled over both arms and blind to the arm: which events each user had, and on how
many days.

In [ ]:
clustered = pre_exposure.add_clusters(
    "behaviour",
    features=[{"metric": "has_event_bulk"}, {"metric": "active_days"}],
    method="kmeans",
    method_args={"n_clusters": 3, "n_init": 4, "random_state": 42},
    scaler="minmax",
)
labels = clustered.get_path_segments("behaviour")
print(
    "recipe shows a truncation before the clustering:", labels.attrs.get("pre_exposure")
)

# Profile: share of each cluster's users with each event, relative to all users.
flags = pre_exposure.get_metrics([{"metric": "has_event_bulk"}])
flags.columns = flags.columns.str.removeprefix("has_event_bulk_")
lift = flags.groupby(labels).mean() / flags.mean()
lift.round(2)

Each cluster has a marker, the event most over-represented in it. We name the clusters after their
markers in code, so the names follow the data and not the K-means label numbers, and we use the
dataset's names for its user types: the `search_filter` cluster is `scanner`, the
`collections_view` (or `wishlist_add`, its tie) cluster is `curator`, and the cluster whose marker is plain browsing is `other`.
Then `add_segment(path_values=...)` carries the labels from the truncated stream back onto the full
one; the `pre_exposure` flag travels with them.

In [ ]:
marker = lift.idxmax(axis=1)
names = {
    "search_filter": "scanner",
    "collections_view": "curator",
    "wishlist_add": "curator",  # ties with collections_view in the curator cluster
}
cluster_names = {label: names.get(event, "other") for label, event in marker.items()}
assert len(set(cluster_names.values())) == 3, "two clusters share a name"
print(marker.to_dict())
print(cluster_names)

stream = stream.add_segment("behaviour", path_values=labels.map(cluster_names))
stream.segment_overview(
    "behaviour",
    metrics=[
        {
            "metric": "has_event",
            "metric_args": {"event": "search_filter"},
            "agg": "mean",
        },
        {
            "metric": "has_event",
            "metric_args": {"event": "collections_view"},
            "agg": "mean",
        },
        {"metric": "has_event", "metric_args": {"event": "quick_add"}, "agg": "mean"},
        {
            "metric": "has_event",
            "metric_args": {"event": "order_success"},
            "agg": "mean",
        },
        {"metric": "active_days", "agg": "mean"},
    ],
).render_static()

In [ ]:
strata = stream.get_ab_test(
    DIFF,
    conversion,
    assignment=assignment,
    strata_col="behaviour",
    strata_pre_exposure=True,  # our argument above; the library records it as declared
)
strata[
    [
        "stratum",
        "group1_paths",
        "group2_paths",
        "group1_share",
        "group2_share",
        "delta",
        "delta_ci_low",
        "delta_ci_high",
        "p_value_adjusted",
        "verdict",
        "stratum_vs_rest_p_value",
        "strata_pre_exposure",
        "strata_pre_exposure_source",
        "exploratory",
    ]
]

In [ ]:
first = strata.iloc[0]
print(f"balance of strata across arms: p = {first['balance_p_value']:.3f}")
print(
    f"heterogeneity (Cochran's Q) over {first['heterogeneity_strata_used']} strata: "
    f"p = {first['heterogeneity_p_value']:.2g}"
)

The flat average was hiding two opposite effects: scanners convert more with `quick_add` and
curators convert less; both signs survive Holm's correction over the strata, and Cochran's Q says
the strata differ. The `other` users, the majority, show no detectable change: their interval (about
-2.5 to +1.8 pp) still allows a small effect in either direction. The `<MISSING>` stratum holds
the silent users, who have no pre-exposure window and so no cluster; they count in the population
but cannot be tested (`insufficient_data`).

We declared the strata pre-exposure (`strata_pre_exposure=True`), so the rows say
`strata_pre_exposure_source="declared"` and are not `exploratory`. The declaration is our argument,
not the library's finding: it holds by construction here, and with real data it rests on the
window above (balanced, and cut before the first add). The balanced shares of the strata across
arms are consistent with it, though balance alone would not prove it (step 9).

**What this proves.** The effect of `quick_add` is not the same for all users: Cochran's Q rejects
equal effects across the strata. **What remains a hypothesis.** *Which* users: the clusters come
from analyst choices (K = 3, the features, the scaler), and the biggest of several stratum effects
is biased upwards, so replicate the split in a new test before targeting anyone. Also the cluster
*names* and what they mean about users, and every *why*: the next steps look for a mechanism, which
paths can suggest but not prove.

## 6. Funnels per cluster

Within a pre-exposure cluster the arms are still randomised, so the overall funnel steps remain
fair comparisons. A loop over the clusters gives one funnel each.

In [ ]:
cluster_funnels = {}
for name in sorted(cluster_names.values()):
    cluster = stream.filter_paths(
        {"in_segment": {"segment_name": "behaviour", "segment_level": name}}
    )
    result = cluster.funnel_data(steps, diff=DIFF, ci_level=0.95, correction="holm")
    last = result["steps"][-1]
    cluster_funnels[name] = {
        "users_treatment": result["funnel1_total_paths"],
        "users_control": result["funnel2_total_paths"],
        "order_delta": last["delta_conversion_rate"],
        "order_ci_low": last["delta_conversion_rate_ci_low"],
        "order_ci_high": last["delta_conversion_rate_ci_high"],
        "order_p_adjusted": last["delta_conversion_rate_p_value_adjusted"],
    }
pd.DataFrame.from_dict(cluster_funnels, orient="index")

In [ ]:
curators = stream.filter_paths(
    {"in_segment": {"segment_name": "behaviour", "segment_level": "curator"}}
)
curators.funnel(steps, diff=DIFF, ci_level=0.95, correction="holm").render_static()

The last step of each cluster's funnel repeats the stratum row of step 5 (silent users have no
cluster, so a cluster's users are exactly its users with events), and the funnel shows where along
the path each cluster's difference builds up: open the curators' funnel above and compare the
steps' whiskers.

**What this proves.** Within each cluster, the overall step deltas with their intervals.
**What remains a hypothesis.** Which step is "the" cause: the deltas of successive overall steps are
not independent, and a conditional step has the selection problem of step 3.

## 7. What curators do differently

Back to the full stream after assignment (the whole log, see step 1), restricted to the curators.
The transition graph in diff mode colours the edges whose weights differ between the arms; the step
matrix centred on `cart` shows what happens just before and after it.

In [ ]:
curators.transition_graph(edge_weight="count", diff=DIFF).render_static()

In [ ]:
_, graph_treatment, graph_control = curators.transition_graph_data(
    edge_weight="count", diff=DIFF
)
print("cart -> cart transitions, curators:")
print("  treatment:", graph_treatment.loc["cart", "cart"])
print("  control:  ", graph_control.loc["cart", "cart"])

In [ ]:
curators.step_matrix(diff=DIFF, path_pattern="cart", step_window=3).render_static()

Treatment curators reopen the cart again and again (`cart -> cart`, a self-edge that control
curators never take), right after a `quick_add`. A plausible reading, and only that: curators
build wishlists and collections, a one-tap add on a tile fills their cart with items they are still
comparing, and the cart becomes a place to review rather than a step to checkout.

**What this shows.** `cart -> cart` transitions among treatment curators and none among control
curators: a description of the paths of users who reached the cart; step 8 tests the loop on the
whole arm. **What remains a hypothesis.** That the loop *causes* the curators' lower conversion. The graph
and the matrix describe the paths of users who looped; they cannot say what these users would have
done without the feature.

## 8. The loop and the order

Two questions, two tools. First, descriptive: among curators who looped, how often did an order
follow, next to curators who simply reached the cart? `get_conversion_rate` gives each rate a
Wilson interval and, in diff mode, the delta between arms.

In [ ]:
loop = {"pattern": "cart->cart->cart"}
loop_to_order = curators.get_conversion_rate(
    [loop, "cart"], "order_success", ci_level=0.95, diff=DIFF
)
loop_to_order[
    [
        "start_anchor",
        "group1_paths_with_start",
        "group1_conversion_rate",
        "group1_conversion_rate_ci_low",
        "group1_conversion_rate_ci_high",
        "group2_paths_with_start",
        "group2_conversion_rate",
        "delta",
        "conditional",
    ]
]

This table is **descriptive**, and the library says so: `conditional=True` on both rows, because
the start anchor is not the path start, so the denominator is whoever reached it, and the
treatment changes who that is. We leave its p-value out of the table for that reason: it would test
a comparison the randomisation does not protect. Loopers exist in treatment only, so the loop row
has no control counterpart (its control rate and delta are NaN: absent, not zero), and comparing
loopers with non-loopers compares users who selected themselves into the loop. Even the `cart` row
compares post-exposure populations: who reaches the cart depends on the arm.

Second, causal: did the treatment change how many users loop? That is a question about the whole
arm, so it goes to `get_ab_test` as an ITT metric, with `matches_pattern` on the loop.

In [ ]:
loop_metric = [
    {
        "name": "cart_loop",
        "metric": "matches_pattern",
        "metric_args": {"pattern": "cart->cart->cart"},
    }
]
loop_itt = stream.get_ab_test(
    DIFF,
    loop_metric,
    assignment=assignment,
    strata_col="behaviour",
    strata_pre_exposure=True,
)
loop_itt[
    [
        "stratum",
        "group1_paths",
        "group2_paths",
        "group1_value",
        "group2_value",
        "delta",
        "delta_ci_low",
        "delta_ci_high",
        "p_value_adjusted",
        "verdict",
        "support_warning",
    ]
]

Only treatment curators loop: about 26 % of them, and no control curator. The support rule pools
the arms (at least `min_events` paths with the event, and as many without it, across both arms), so
a behaviour that exists in one arm only still gets a p-value; the row says so with
`support_warning="one_arm_only"`. Read the curators' `increase` for what it is: a behaviour the
treatment introduced. The p-value says it is not noise; the size, 26 % against 0 %, is not an
effect on an existing behaviour, because control has no loop for the treatment to change. The rows of the
other strata have no looper in either arm, so their support is zero and they stay
`insufficient_data`. The loop is a product of the treatment, concentrated in curators.

**What this proves.** `quick_add` creates the cart loop, and only among curators. **What remains
a hypothesis.** That the loop is how `quick_add` costs curators their orders. The experiment
randomised the button, not the loop; to test the mechanism, run the next experiment on it (for
example, `quick_add` that skips the cart for curator-like users), with conversion as the metric.

## 9. Path length: pre-exposure strata, and why post-exposure ones mislead

Another common cut is path length. Computed on the **pre-exposure** window (a median split of the
events before the first add), it is fixed before the treatment can act.

In [ ]:
length_bins = {
    "metric": {"metric": "length"},
    "quantiles": 2,
    "segment_levels": ["short", "long"],
}
pre_length = pre_exposure.add_segment(
    "length_bin", metric_bins=length_bins
).get_path_segments("length_bin")
by_length = stream.add_segment("length_bin", path_values=pre_length).get_ab_test(
    DIFF,
    conversion,
    assignment=assignment,
    strata_col="length_bin",
    strata_pre_exposure=True,  # the same window as the clusters
)
by_length[
    [
        "stratum",
        "group1_share",
        "group2_share",
        "delta",
        "delta_ci_low",
        "delta_ci_high",
        "p_value_adjusted",
        "verdict",
        "heterogeneity_p_value",
        "strata_pre_exposure_source",
    ]
]

The sign flips with pre-exposure length: users who browse little before their first add convert
less with `quick_add`, users who browse a lot convert more, both after Holm.

Now the tempting mistake: the same median split on the **whole** path. Length after exposure is
itself an outcome (the cart loop of step 7 makes paths longer), so the strata are selected by the
treatment. Nothing is declared this time, and the stream's recipe has no truncation, so the
library warns and flags the rows `strata_pre_exposure=False` (`strata_pre_exposure_source=
"not_pre_exposure"`) and `exploratory=True`.

In [ ]:
post_length = stream.add_segment("length_bin", metric_bins=length_bins).get_ab_test(
    DIFF, conversion, assignment=assignment, strata_col="length_bin"
)
post_length[
    [
        "stratum",
        "group1_share",
        "group2_share",
        "balance_p_value",
        "delta",
        "p_value_adjusted",
        "verdict",
        "strata_pre_exposure",
        "strata_pre_exposure_source",
        "exploratory",
    ]
]

The post-exposure split tells a different story from the pre-exposure one, and its share-balance
p-value does not raise an alarm: the stratum shares can be balanced while the people in them are
not comparable. `balance_p_value` checks shares only; it is not a guard against post-exposure
strata. The guard is how the stratum was computed. `strata_pre_exposure` records what the library
knows about that, the recipe or your declaration; whether the cut sits before exposure is your
argument, as in step 5.

How does the length split relate to the clusters of step 5? Both are pre-exposure labels, so a
crosstab of the two is fair to read.

In [ ]:
pd.crosstab(
    stream.get_path_segments("behaviour"),
    pre_length.reindex(stream.get_path_segments("behaviour").index),
    normalize="index",
)

The two splits overlap only partly: each cluster has both short and long users. Whether length
carries an effect of its own beyond the clusters, or the clusters beyond length, needs strata of
both at once, and at this sample size such cells are thin: an exploratory question.

**What this proves.** The pre-exposure length split is a valid subgroup comparison with opposite
signs. **What remains a hypothesis.** How it relates to the clusters, and any mechanism.

## 10. Planted truth, a summary, and a few paths

The dataset is synthetic, so we can check the analysis against the planted user types
(`with_truth=True`, used here and nowhere else).

In [ ]:
truth = (
    datasets.load_ab_test(as_dataframe=True, with_truth=True)
    .groupby("user_id")["user_type"]
    .first()
)
recovered = stream.get_path_segments("behaviour")
print(
    f"adjusted Rand index, clusters vs planted types: {adjusted_rand_score(truth, recovered.reindex(truth.index)):.3f}"
)
pd.crosstab(recovered.reindex(truth.index), truth)

The blind clusters recover the planted types closely (the crosstab shows where they disagree), and
the names given from the marker events match the planted ones. That is the best case: real
behaviour types overlap more, and a clustering always returns *some* groups. Step 5's test shows
that the effect differs between users; which users is a hypothesis until a replication on new data
shows the same strata reacting the same way.

The summary below is assembled from the results above; nothing in it is typed by hand.

In [ ]:
def ci_text(row, scale=100, unit=" pp"):
    return (
        f"{row['delta'] * scale:+.2f}{unit} [{row['delta_ci_low'] * scale:+.2f}, "
        f"{row['delta_ci_high'] * scale:+.2f}], {row['verdict']}"
    )


conv_row = scorecard.set_index("metric").loc["conversion"]
conv_28d = scorecard_28d.set_index("metric").loc["conversion"]
aov_row = scorecard.set_index("metric").loc["aov"]
strata_rows = strata.set_index("stratum")
length_rows = by_length.set_index("stratum")
payment_step = funnel_table.loc["payment"]
looped = loop_to_order.iloc[0]
loop_share = loop_itt.set_index("stratum").loc["curator"]
summary = pd.DataFrame(
    [
        (
            "get_sample_ratio",
            "arms match the 50/50 design",
            f"p = {srm['p_value']:.3f}",
            "design check",
        ),
        (
            "get_ab_test",
            "conversion, whole window (primary read)",
            ci_text(conv_row),
            "randomised; declared in step 2",
        ),
        (
            "get_ab_test(horizon='28D')",
            "conversion, first 28 days after assignment (secondary read)",
            ci_text(conv_28d),
            "randomised; declared in step 2",
        ),
        ("get_ab_test", "average order value", ci_text(aov_row, 1, ""), "randomised"),
        (
            "funnel(ci_level=, correction=)",
            "overall steps, min adjusted p",
            f"{funnel_table['delta_conversion_rate_p_value_adjusted'].min():.3f}",
            "randomised",
        ),
        (
            "funnel",
            "checkout_start -> payment (conditional)",
            f"{payment_step['delta_step_conversion_rate'] * 100:+.2f} pp "
            f"[{payment_step['delta_step_conversion_rate_ci_low'] * 100:+.2f}, "
            f"{payment_step['delta_step_conversion_rate_ci_high'] * 100:+.2f}], no p",
            "descriptive",
        ),
        (
            "get_ab_bootstrap(contrast=)",
            "effect days 0-13 minus days 28-41 (horizon 42D)",
            f"{point * 100:+.2f} pp [{low * 100:+.2f}, {high * 100:+.2f}]",
            "sensitivity check, not a third estimand; cause is a hypothesis",
        ),
        (
            "get_ab_test(strata_col=)",
            "scanners",
            ci_text(strata_rows.loc["scanner"]),
            "randomised within stratum",
        ),
        (
            "get_ab_test(strata_col=)",
            "curators",
            ci_text(strata_rows.loc["curator"]),
            "randomised within stratum",
        ),
        (
            "get_ab_test(strata_col=)",
            "other users",
            ci_text(strata_rows.loc["other"]),
            "randomised within stratum",
        ),
        (
            "transition_graph / step_matrix",
            "cart -> cart, curators",
            f"{graph_treatment.loc['cart', 'cart']} vs {graph_control.loc['cart', 'cart']} transitions",
            "descriptive",
        ),
        (
            "get_conversion_rate(diff=)",
            "loop -> order, curators",
            f"treatment {looped['group1_conversion_rate'] * 100:.1f} % "
            f"[{looped['group1_conversion_rate_ci_low'] * 100:.1f}, "
            f"{looped['group1_conversion_rate_ci_high'] * 100:.1f}]; "
            f"control has {looped['group2_paths_with_start']} loopers",
            "descriptive",
        ),
        (
            "get_ab_test(matches_pattern)",
            "share of curators who loop",
            f"{loop_share['group1_value'] * 100:.1f} % vs {loop_share['group2_value'] * 100:.1f} %, "
            f"{loop_share['verdict']}",
            f"randomised; support_warning={loop_share['support_warning']!r}: a behaviour "
            "the treatment introduced; the p-value says it is not noise, the size is not "
            "an effect on an existing behaviour",
        ),
        (
            "get_ab_test(strata_col=)",
            "short pre-exposure paths",
            ci_text(length_rows.loc["short"]),
            "randomised within stratum",
        ),
        (
            "get_ab_test(strata_col=)",
            "long pre-exposure paths",
            ci_text(length_rows.loc["long"]),
            "randomised within stratum",
        ),
        (
            "-",
            "the loop costs curators their orders",
            "-",
            "hypothesis: next experiment",
        ),
    ],
    columns=["call", "question", "result", "evidence"],
)
summary

Before presenting, read a few raw paths: they keep the story honest about what users actually do.
`sample_paths` draws users at random; here two curators per arm among those who reached the
cart, one line per session.

In [ ]:
def show_paths(sample):
    frame = sample.to_dataframe()
    for user_id, user_events in frame.groupby("user_id", sort=False):
        print(f"user {user_id} ({user_events['exp_group'].iloc[0]}):")
        for session_id, session in user_events.groupby("session_id", sort=False):
            print("   ", " -> ".join(session["event"]))


for arm in ["treatment", "control"]:
    in_arm = curators.filter_paths(
        [
            {"in_segment": {"segment_name": "exp_group", "segment_level": arm}},
            {"has_event": "cart"},
        ]
    )
    show_paths(in_arm.sample_paths(n=2, random_state=7))

**The readout.** Over the whole observation period (the primary read) the effect of `quick_add`
on conversion is inconclusive: the interval includes zero and still allows a small lift or a small
loss. Within each user's first 28 days (the secondary read) it raised conversion, orders and
revenue. By time since assignment, the lift came in the first two weeks and reversed to a small
decrease after four weeks (-1.5 pp, adjusted p 0.023). The effect differs between users: in this
test it helped users who search with filters (scanners) and hurt users who build collections
(curators), whose carts turn into a review loop; which users, and the loop as the cause, are
hypotheses to replicate. The launch decision is a product call on these facts; the next experiment
is on the mechanism (the loop), not on the button again.

Further reading: the [A/B testing guide](https://retentioneering.com/docs/ab-testing) for the
mental model and the pitfalls, and the
[`get_ab_test` reference](https://retentioneering.com/docs/tools/get-ab-test) for every output
column.